# 01 – Clean California data
## Step 1 – Load and inspect
Purpose: remove duplicates, convert dates, flag impossible dates, add helper flags.
Input: data/raw/california/chemicals_in_cosmetics.csv
Output: data/clean/california.csv

In [1]:
import pandas as pd

# Raw data: one row = one chemical reported in one product (CDPH, 2009–2020)
ca = pd.read_csv("../data/raw/california/chemicals_in_cosmetics.csv")

# Overview: column names, non-missing counts and data types
ca.info()

<class 'pandas.DataFrame'>
RangeIndex: 114635 entries, 0 to 114634
Data columns (total 22 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   CDPHId                  114635 non-null  int64  
 1   ProductName             114635 non-null  str    
 2   CSFId                   80662 non-null   float64
 3   CSF                     80237 non-null   str    
 4   CompanyId               114635 non-null  int64  
 5   CompanyName             114635 non-null  str    
 6   BrandName               114408 non-null  str    
 7   PrimaryCategoryId       114635 non-null  int64  
 8   PrimaryCategory         114635 non-null  str    
 9   SubCategoryId           114635 non-null  int64  
 10  SubCategory             114635 non-null  str    
 11  CasId                   114635 non-null  int64  
 12  CasNumber               108159 non-null  str    
 13  ChemicalId              114635 non-null  int64  
 14  ChemicalName            114635 

**Findings:**
- 114,635 rows, 22 columns.
- Missing values in 6 columns. Missing DiscontinuedDate / ChemicalDateRemoved mean
  "still on the market" / "not removed" – these are information, not errors.
- 6,476 rows have no CAS number; they cannot be matched to the EU lists.
- All 6 date columns are text and must be converted to real dates.
- Most rows without CAS are chemical groups (crystalline silica, mineral oils, retinol esters); they stay in the data but can’t be matched to EU lists by CAS.


## Step 2 – Duplicates  
Exact duplicate rows would count the same report twice.

In [2]:
# Count rows that are exact copies of an earlier row (all 22 columns identical)
ca.duplicated().sum()

np.int64(254)

In [3]:
# Show all copies of duplicated rows, sorted so the pairs sit next to each other
ca[ca.duplicated(keep=False)].sort_values("CDPHId").head(6)

,CDPHId,ProductName,CSFId,CSF,CompanyId,CompanyName,BrandName,PrimaryCategoryId,PrimaryCategory,SubCategoryId,...,CasNumber,ChemicalId,ChemicalName,InitialDateReported,MostRecentDateReported,DiscontinuedDate,ChemicalCreatedAt,ChemicalUpdatedAt,ChemicalDateRemoved,ChemicalCount
15086,3488,CHI Scentuals Mango Melon Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1
15087,3488,CHI Scentuals Mango Melon Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1
15093,3494,CHI Scentuals Lavender Latte Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1
15094,3494,CHI Scentuals Lavender Latte Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1
15115,3506,CHI Scentuals Coconut Lime Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1
15116,3506,CHI Scentuals Coconut Lime Hand and Body Wash,NaN,NaN,120,"Farouk Systems, Inc.",CHI,6,Bath Products,159,...,0,0,Trade Secret,10/12/2009,10/12/2009,NaN,10/12/2009,10/12/2009,NaN,1


In [4]:
# Remove the 254 exact duplicate rows
ca = ca.drop_duplicates()
ca.shape

(114381, 22)

**finding:**
- “254 duplicates removed → 114,381 rows. Rows with the same CDPHId are not duplicates (one row = product × shade × chemical)

## Step 3 – Convert dates 
Dates are stored as text (MM/DD/YYYY); real dates are needed to group by year.

In [5]:
# The six date columns, stored as text
date_cols = ["InitialDateReported", "MostRecentDateReported", "DiscontinuedDate",
             "ChemicalCreatedAt", "ChemicalUpdatedAt", "ChemicalDateRemoved"]

ca[date_cols].head()

,InitialDateReported,MostRecentDateReported,DiscontinuedDate,ChemicalCreatedAt,ChemicalUpdatedAt,ChemicalDateRemoved
0,06/17/2009,08/28/2013,02/01/2011,07/09/2009,07/09/2009,NaN
1,07/01/2009,07/01/2009,NaN,07/01/2009,07/01/2009,NaN
2,07/01/2009,07/01/2009,NaN,07/02/2009,07/02/2009,NaN
3,07/09/2009,08/28/2013,NaN,07/09/2009,07/09/2009,NaN
4,07/09/2009,08/28/2013,02/01/2011,07/09/2009,07/09/2009,NaN


In [6]:
# Convert each date column from text (MM/DD/YYYY) to a real date;
# unreadable values become NaT (missing) instead of stopping the code
for col in date_cols:
    ca[col] = pd.to_datetime( ca[col] , format="%m/%d/%Y", errors="coerce")

In [7]:
  # Check: all six columns are now real dates
  ca[date_cols].dtypes

InitialDateReported       datetime64[us]
MostRecentDateReported    datetime64[us]
DiscontinuedDate          datetime64[us]
ChemicalCreatedAt         datetime64[us]
ChemicalUpdatedAt         datetime64[us]
ChemicalDateRemoved       datetime64[us]
dtype: object

**Findings:**
- All 6 date columns converted to datetime (format MM/DD/YYYY); no values lost in the conversion.

In [8]:
# Renumber rows 0..114380 after removing duplicates
ca = ca.reset_index(drop=True)

## Step 4 – Impossible dates 
The programme ran 2009–2020; dates outside are errors.

In [9]:
# Earliest and latest date per column, to spot impossible dates
ca[date_cols].agg(["max", "min"])

,InitialDateReported,MostRecentDateReported,DiscontinuedDate,ChemicalCreatedAt,ChemicalUpdatedAt,ChemicalDateRemoved
max,2020-06-23,2020-06-24,2020-06-12,2020-06-23,2020-06-24,2104-05-02
min,2009-06-17,2009-07-01,2001-01-01,2009-07-01,2009-07-01,2000-01-01


## Step 5 – Helper flags 
Yes/no columns for the analysis (H1 uses is_removed).

In [10]:
# True if the chemical was removed from the product (a removal date exists)
ca["is_removed"] = ca["ChemicalDateRemoved"].notna()
ca["is_removed"].sum()

np.int64(2985)

In [11]:
# Removal dates outside 2009–2020 are impossible (e.g. 2103, 2000).
# The removal itself is real (is_removed stays True), but the date is unreliable -> set to missing
bad = (ca["ChemicalDateRemoved"] < "2009-01-01") | (ca["ChemicalDateRemoved"] > "2020-12-31")
print(bad.sum())
ca.loc[bad, "ChemicalDateRemoved"] = pd.NaT

118


In [12]:
ca["ChemicalDateRemoved"].agg(["min", "max"])

min   2009-01-01
max   2020-05-14
Name: ChemicalDateRemoved, dtype: datetime64[us]

**Findings:**
- 118 removal dates were impossible (2000, 2006, 2103, 2104; most likely typing errors such as 2103 for 2013). The dates were set to missing, but the rows keep is_removed = True, because the removal itself was reported. DiscontinuedDate has 739 dates before 2009; these are plausible, because the law covers products sold since 2007, so they were kept.

In [13]:
# Flag: was the whole product discontinued?
ca["is_discontinued"] = ca["DiscontinuedDate"].notna()
print(ca["is_discontinued"].sum())   # 12873

# Flag: is the reported chemical titanium dioxide? (81% of all rows, analysed separately)
ca["is_tio2"] = ca["ChemicalName"].str.contains("Titanium dioxide", case=False)
print(ca["is_tio2"].mean())          # about 0.81

12873
0.8179942472963166


## Step 6 – Clean CAS numbers
CAS numbers are the key for matching chemicals to the EU lists, so they must be in the exact
format "digits-digits-digit" (e.g. 13463-67-7). Spaces, placeholders and broken numbers are fixed or set to missing.

In [14]:
# Remove spaces before/after each CAS number (93 rows affected)
ca["CasNumber"] = ca["CasNumber"].str.strip()
# Should be 0: among filled CAS numbers, none differs from its stripped version
(ca["CasNumber"].notna() & (ca["CasNumber"] != ca["CasNumber"].str.strip())).sum()

np.int64(0)

In [15]:
# Some companies hid the chemical's identity: name "Trade Secret", CAS "0" (not a real number)
ca["is_trade_secret"] = ca["ChemicalName"].str.contains("Trade Secret", case=False)
print(ca["is_trade_secret"].sum())                         # expected 416
ca.loc[ca["CasNumber"] == "0", "CasNumber"] = pd.NA        # "0" is a placeholder -> missing

416


**Findings:**
 - 416 rows report the chemical as “Trade Secret” with CAS “0” (a placeholder, not a real CAS number). Flagged as is_trade_secret; CAS set to missing.

In [16]:
# A valid CAS number looks like 2–7 digits, hyphen, 2 digits, hyphen, 1 digit
valid = ca["CasNumber"].str.fullmatch(r"\d{2,7}-\d{2}-\d", na=False)

# Look at the broken ones before changing them
broken = ca["CasNumber"].notna() & ~valid
print(broken.sum())                                        # expected 52
ca.loc[broken, "CasNumber"].value_counts()

52


CasNumber
CAS # 79-81-    9
13463-67-       9
11103-57-438    7
201-228-5       3
RN:  93�15�2    3
93 15 2         3
7440-43-10      2
79812           2
CAS #79-81-2    2
13463677        2
RN: 13463-67    2
79-818-2        1
11103-57-538    1
CAS#79-81-2     1
79-8-12         1
101--54-2       1
asdf            1
RN:  13463 6    1
50-78-25        1
Name: count, dtype: int64

In [17]:
# Broken CAS numbers can't be matched reliably -> set to missing (documented, not guessed)
ca.loc[broken, "CasNumber"] = pd.NA
ca["CasNumber"].isna().sum()                               # expected 6943

np.int64(6943)

In [18]:
# How many different products, companies and chemicals?
print("Products: ", ca["CDPHId"].nunique())
print("Companies:", ca["CompanyId"].nunique())
print("Chemicals:", ca["ChemicalName"].nunique())

Products:  36972
Companies: 635
Chemicals: 123


**Findings:**
 - The cleaned data covers 36,972 products from 635 companies, with 123 different reported chemicals.

In [19]:
# Save the cleaned data for the next notebooks
ca.to_csv("../data/clean/california.csv", index=False)
ca.shape   # expected (114381, 25)

(114381, 26)

## Summary

| Step | Result |
|---|---|
| Loaded raw data | 114,635 rows × 22 columns |
| Removed exact duplicates | 254 rows → 114,381 rows |
| Converted 6 date columns (MM/DD/YYYY → datetime) | no values lost |
| Impossible removal dates (2000, 2006, 2103, 2104) | 118 set to missing; rows keep is_removed = True |
| Discontinued dates before 2009 | 739 kept (plausible: the law covers products sold since 2007) |
| Flag is_removed | 2,985 rows (chemical removed from the product) |
| Flag is_discontinued | 12,873 rows (product discontinued) |
| Flag is_tio2 | 81.8% of rows (titanium dioxide, analysed separately) |
| Rows without CAS number | about 6,475, mostly chemical groups; kept, but not matchable to EU lists by CAS |
| Cleaned CAS numbers | 93 with spaces fixed; 416 "Trade Secret" (CAS 0) and 52 broken numbers set to missing → 6,943 rows without a valid CAS |
| Flag is_trade_secret | 416 rows (chemical identity hidden by the company) |
| Coverage | 36,972 products · 635 companies · 123 chemicals |

**Output:** data/clean/california.csv (114,381 rows × 25 columns)

**Next (02_clean_cosing):** clean CAS numbers and match them to the EU annexes.